# BBBC031: training Stochastic Mixture (NB=3 e NB=7) su Colab

Questo notebook addestra **Extended Stochastic Mixture NCA** su un'immagine BBBC031 per due dimensioni di vicinato (3 e 7), adatto a Google Colab.

**Prima di eseguire:**
1. Carica il dataset BBBC031 (cartella `BBBC031_v1_dataset` con sottocartella `Images/`) e il CSV ground truth su Google Drive, oppure caricali nella sessione Colab.
2. Imposta sotto i path `DATASET_DIR` e `CSV_PATH` (e opzionalmente `REPO_URL` se il repo è su GitHub).

## 1. Setup: clone repo e dipendenze

In [ ]:
# Clone del repo (cambia REPO_URL con il tuo repo se necessario)
REPO_URL = "https://github.com/luigidaddario/MNCA.git"  # oppure il path del tuo fork

!git clone --depth 1 {REPO_URL} /content/MNCA
%cd /content/MNCA
!pip install -q -r requirements.txt

## 2. Path dei dati (Drive o upload)

Scegli **una** delle due opzioni: monta Drive e imposta i path, oppure usa i path dove hai caricato i file nella sessione.

In [ ]:
# Opzione A: monta Google Drive e imposta i path alla cartella BBBC031
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = "/content/drive/MyDrive/BBBC031_v1_dataset"   # cartella con Images/ e Masks/
CSV_PATH    = "/content/drive/MyDrive/BBBC031_v1_DatasetGroundTruth.csv"  # CSV ground truth (sep=";")

# Opzione B: se hai caricato i file in /content/ (es. zip estratto):
# DATASET_DIR = "/content/BBBC031_v1_dataset"
# CSV_PATH    = "/content/BBBC031_v1_DatasetGroundTruth.csv"

import os
assert os.path.isdir(DATASET_DIR), f"Dataset non trovato: {DATASET_DIR}"
assert os.path.isfile(CSV_PATH), f"CSV non trovato: {CSV_PATH}"
assert os.path.isdir(os.path.join(DATASET_DIR, "Images")), "Manca la sottocartella Images/"
print("Path dati OK.")

## 3. Training: Stochastic Mixture con **neighborhood size = 3**

In [ ]:
# Esecuzione in-process così la barra di avanzamento (tqdm) si vede nel notebook
import sys
import os
os.chdir("/content/MNCA")
if "/content/MNCA" not in sys.path:
    sys.path.insert(0, "/content/MNCA")

sys.argv = [
    "train_bbbc031_mnca.py",
    "--dataset_dir", DATASET_DIR,
    "--csv_path", CSV_PATH,
    "--stochastic", "--neighborhood_size", "3",
    "--checkpoint_path", "models/bbbc031_stochastic_NB3.pth",
    "--total_steps", "4000", "--device", "cuda",
]
from experiments.train_bbbc031_mnca import main
main()

## 4. Training: Stochastic Mixture con **neighborhood size = 7**

In [ ]:
os.chdir("/content/MNCA")
sys.argv = [
    "train_bbbc031_mnca.py",
    "--dataset_dir", DATASET_DIR,
    "--csv_path", CSV_PATH,
    "--stochastic", "--neighborhood_size", "7",
    "--checkpoint_path", "models/bbbc031_stochastic_NB7.pth",
    "--total_steps", "4000", "--device", "cuda",
]
from experiments.train_bbbc031_mnca import main
main()

## 5. (Opzionale) Figure di confronto e download

Le figure vengono salvate in `thesis-latex/figs/bbbc031/` durante il training. Qui generiamo le figure del demo per NB=3 e NB=7 e le mostriamo.

In [ ]:
os.chdir("/content/MNCA")

for nb, ckpt in [("3", "models/bbbc031_stochastic_NB3.pth"), ("7", "models/bbbc031_stochastic_NB7.pth")]:
    subprocess.run([
        "python", "experiments/bbbc031_mnca_demo.py",
        "--dataset_dir", DATASET_DIR, "--csv_path", CSV_PATH,
        "--checkpoint", ckpt, "--neighborhood_size", nb, "--stochastic",
        "--out_dir", "thesis-latex/figs/bbbc031", "--num_steps", "20",
    ], check=True)

In [ ]:
from IPython.display import Image, display

fig_dir = "/content/MNCA/thesis-latex/figs/bbbc031"
for name in ["bbbc031_gt_vs_mnca_NB3_stochastic.png", "bbbc031_gt_vs_mnca_NB7_stochastic.png"]:
    path = os.path.join(fig_dir, name)
    if os.path.isfile(path):
        display(Image(path, width=500))
    else:
        print("Non trovata:", path)

In [ ]:
# Scarica i checkpoint e le figure (zip)
!cd /content/MNCA && zip -r /content/bbbc031_stochastic_outputs.zip models/bbbc031_stochastic_NB3.pth models/bbbc031_stochastic_NB7.pth thesis-latex/figs/bbbc031/*.png 2>/dev/null || true
from google.colab import files
files.download("/content/bbbc031_stochastic_outputs.zip")